# 05. Speculative Decoding：Draft、Verify、Accept

> 面向即将加入 SGLang 开源社区的开发者。建议从仓库根目录启动 Jupyter，
> 或者在 notebook 第一格运行路径检查。本文以本地源码为主，线上文档为索引。
> Markdown 中的源码摘录会插入 `# 注：...` 行内讲解；可执行代码 cell 则保持可运行。


In [ ]:
from pathlib import Path
import ast, inspect, re, textwrap

def find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for candidate in [p, *p.parents]:
        if (candidate / "python" / "sglang").exists() and (candidate / "docs").exists():
            return candidate
    raise RuntimeError("没有找到 SGLang 仓库根目录，请从仓库内启动 notebook。")

ROOT = find_repo_root()
print(ROOT)

def read_rel(path):
    return (ROOT / path).read_text()

def show_lines(path, start, end):
    lines = read_rel(path).splitlines()
    for i in range(start, min(end, len(lines)) + 1):
        print(f"{i:4d}: {lines[i-1]}")

def find_defs(path, names=None):
    tree = ast.parse(read_rel(path))
    rows = []
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            if names is None or node.name in names:
                rows.append((node.lineno, type(node).__name__, node.name))
    return sorted(rows)


## 统一心智模型

Speculative decoding 的共同目标是减少 target model 的 decode 次数：

1. draft 产生多个候选 token。
2. target 一次 verify 这些候选。
3. accept/reject 决定提交几个 token。
4. KV cache、seq len、输出 token、采样状态一起前进。

不同算法的差异在 draft 来源：

- `EAGLE` / `EAGLE3`：用 draft model / hidden states 预测树状候选。
- `STANDALONE`：小 draft model 独立生成候选。
- `NGRAM`：从历史 ngram 中检索候选，不需要 draft KV。
- `DFLASH` / `FROZEN_KV_MTP`：更新的内部路径，围绕 target hidden / frozen KV / block verify 做优化。


In [ ]:
show_lines("python/sglang/srt/speculative/spec_info.py", 20, 120)


### `SpeculativeAlgorithm`：算法能力比 enum 名字更重要

```python
# python/sglang/srt/speculative/spec_info.py:20-120
  20:     from sglang.srt.managers.overlap_utils import FutureMap
  21:     from sglang.srt.managers.schedule_batch import ScheduleBatch
  22:     from sglang.srt.managers.tp_worker import TpModelWorker
  23:     from sglang.srt.server_args import ServerArgs
  24:     from sglang.srt.speculative.base_spec_worker import BaseSpecWorker
  25:     from sglang.srt.speculative.ngram_worker import NGRAMWorker
  26: 
  27: 
  28: class SpeculativeAlgorithm(Enum):
  29:     """Builtin speculative decoding algorithms. Plugin-registered ones are
  30:     ``CustomSpecAlgo`` instances; ``from_string`` returns either type, and
  31:     both expose the same ``is_*()`` / ``create_worker`` interface so callers
  32:     dispatch uniformly without isinstance checks.
  33:     """
  34: 
  35:     DFLASH = auto()
  36:     EAGLE = auto()
  37:     EAGLE3 = auto()
  38:     FROZEN_KV_MTP = auto()
  39:     STANDALONE = auto()
  40:     NGRAM = auto()
  41:     NONE = auto()
  42: 
  43:     @classmethod
  44:     def from_string(
  45:         cls, name: Optional[str]
  46:     ) -> Union[SpeculativeAlgorithm, CustomSpecAlgo]:
  47:         if name is None:
  48:             return cls.NONE
  49:         upper = name.upper()
  50:         try:
  51:             return cls[upper]
  52:         except KeyError:
  53:             pass
  54:         spec = _get_registered_spec(upper)
      # 注：`spec` 接收 `_get_registered_spec` 的结果：查询插件注册的 speculative algorithm。
  55:         if spec is not None:
  56:             return spec
  57:         raise ValueError(f"Unknown speculative algorithm name: {name}")
  58: 
  59:     @classmethod
  60:     def register(
  61:         cls,
  62:         name: str,
  63:         *,
  64:         supports_overlap: bool = False,
  65:         validate_server_args: Optional[ServerArgsValidator] = None,
  66:         spec_class: Type[CustomSpecAlgo] = CustomSpecAlgo,
  67:     ) -> Callable[[WorkerFactory], WorkerFactory]:
  68:         """Decorator to register a plugin speculative algorithm. The factory
  69:         takes ``server_args`` and returns the worker class. Pass a
  70:         ``CustomSpecAlgo`` subclass via ``spec_class`` to override any
  71:         ``is_*()`` / ``create_worker`` method.
  72: 
  73:         Example:
  74:             @SpeculativeAlgorithm.register("MY_SPEC", supports_overlap=False)
  75:             def _factory(server_args):
  76:                 return MySpecWorker
  77:         """
  78:         return _register_algorithm(
  79:             name,
  80:             supports_overlap=supports_overlap,
  81:             validate_server_args=validate_server_args,
  82:             spec_class=spec_class,
  83:         )
  84: 
  85:     def is_some(self) -> bool:
  86:         return self != SpeculativeAlgorithm.NONE
  87: 
  88:     def is_none(self) -> bool:
  89:         return self == SpeculativeAlgorithm.NONE
  90: 
  91:     def is_speculative(self) -> bool:
  92:         return self != SpeculativeAlgorithm.NONE
  93: 
  94:     def is_eagle(self) -> bool:
  95:         # FIXME(kpham_sgl): Remove FROZEN_KV_MTP here once we
  96:         # have established support for it in the scheduler.
  97:         return self in (
  98:             SpeculativeAlgorithm.EAGLE,
  99:             SpeculativeAlgorithm.EAGLE3,
 100:             SpeculativeAlgorithm.FROZEN_KV_MTP,
 101:         )
 102: 
 103:     def is_eagle3(self) -> bool:
 104:         return self == SpeculativeAlgorithm.EAGLE3
 105: 
 106:     def is_frozen_kv_mtp(self) -> bool:
 107:         return self == SpeculativeAlgorithm.FROZEN_KV_MTP
 108: 
 109:     def is_dflash(self) -> bool:
 110:         return self == SpeculativeAlgorithm.DFLASH
 111: 
 112:     def is_standalone(self) -> bool:
 113:         return self == SpeculativeAlgorithm.STANDALONE
 114: 
 115:     def is_ngram(self) -> bool:
 116:         return self == SpeculativeAlgorithm.NGRAM
 117: 
 118:     def supports_target_verify_for_draft(self) -> bool:
 119:         return self.is_dflash()
 120: 
```

**读这段时抓住：**

- `from_string` 同时解析内建算法和插件注册算法，所以调用方不能假设返回值一定是 Enum。
- `is_eagle` 把 EAGLE、EAGLE3、FROZEN_KV_MTP 归入同一族，是因为 scheduler 视角上它们共享 draft hidden/KV 约束。
- `has_draft_kv` 区分 NGRAM 这种不写 draft KV 的算法，直接影响每步 KV reserve 估算。
- `need_topk` 决定 target/draft forward 是否需要额外 top-k 信息。


## `SpeculativeAlgorithm` 是调度分发的统一接口

注意这里不是简单 enum。插件也可以注册 custom speculative algorithm，但必须 duck-type 出相同的 `is_*` / `supports_*` 接口。这样 scheduler/model runner 可以统一调用。


In [ ]:
show_lines("python/sglang/srt/speculative/spec_registry.py", 24, 125)
show_lines("python/sglang/srt/speculative/spec_info.py", 150, 225)


### `CustomSpecAlgo`：插件算法必须 duck-type 内建接口

```python
# python/sglang/srt/speculative/spec_registry.py:24-118
  24: class CustomSpecAlgo:
  25:     """A plugin-registered speculative algorithm. Duck-types
  26:     ``SpeculativeAlgorithm`` enum values (same ``is_*()`` / ``create_worker``
  27:     interface).
  28: 
  29:     Plugins may subclass this to override any ``is_*()`` / ``supports_*()`` /
  30:     ``create_worker`` method (e.g. to integrate with builtin-specific
  31:     branches like ``if spec_algorithm.is_eagle():`` in scheduler /
  32:     model_runner). Pass the subclass via ``spec_class=...`` at registration.
  33: 
  34:     Defaults: all ``is_*()`` return ``False`` except ``is_speculative``.
  35: 
  36:     ``supports_overlap=False`` is deprecated: the spec V1 worker path has been
  37:     removed, so such algorithms run on the V2 scheduler schema with overlap
  38:     disabled (synchronous). Migrate plugin workers to the V2 schema and
  39:     overlap scheduling.
  40:     """
  41: 
  42:     def __init__(
  43:         self,
  44:         name: str,
  45:         factory: WorkerFactory,
  46:         *,
  47:         supports_overlap: bool = False,
  48:         validate_server_args: Optional[ServerArgsValidator] = None,
  49:     ):
  50:         self.name = name
  51:         self.factory = factory
  52:         self.supports_overlap = supports_overlap
  53:         self.validate_server_args = validate_server_args
  54: 
  55:     def __repr__(self) -> str:
  56:         return f"CustomSpecAlgo({self.name!r})"
  57: 
  58:     def is_some(self) -> bool:
  59:         return True
  60: 
  61:     def is_none(self) -> bool:
  62:         return False
  63: 
  64:     def is_speculative(self) -> bool:
  65:         return True
  66: 
  67:     def is_eagle(self) -> bool:
  68:         return False
  69: 
  70:     def is_eagle3(self) -> bool:
  71:         return False
  72: 
  73:     def is_frozen_kv_mtp(self) -> bool:
  74:         return False
  75: 
  76:     def is_dflash(self) -> bool:
  77:         return False
  78: 
  79:     def is_standalone(self) -> bool:
  80:         return False
  81: 
  82:     def is_ngram(self) -> bool:
  83:         return False
  84: 
  85:     def supports_target_verify_for_draft(self) -> bool:
  86:         return False
  87: 
  88:     def has_draft_kv(self) -> bool:
  89:         # Conservative default: the larger KV reserve.
  90:         return True
  91: 
  92:     def handle_server_args(self, server_args: ServerArgs) -> None:
  93:         pass
  94: 
  95:     def create_worker(self, server_args: ServerArgs) -> Type:
  96:         if not server_args.disable_overlap_schedule and not self.supports_overlap:
  97:             raise ValueError(
  98:                 f"Speculative algorithm {self.name} does not support overlap scheduling."
  99:             )
 100:         if not self.supports_overlap:
 101:             # Reached only when overlap is disabled, so the algorithm really
 102:             # does run synchronously on the V2 schema below.
 103:             logger.warning(
 104:                 "Speculative algorithm %s is registered with "
 105:                 "supports_overlap=False, which is deprecated: the spec V1 "
 106:                 "worker path has been removed, and the algorithm now runs on "
 107:                 "the V2 scheduler schema with overlap disabled (synchronous). "
 108:                 "Migrate the plugin worker to support overlap scheduling.",
 109:                 self.name,
 110:             )
 111:         return self.factory(server_args)
      # 注：关键调用：`self.factory` - 调用插件提供的 worker factory，得到自定义 speculative worker 类。
 112: 
 113:     def get_num_tokens_per_bs_for_target_verify(
 114:         self, num_draft_tokens: int, is_draft_worker: bool
 115:     ) -> int:
 116:         # FIXME: Remove this after the forward mode refactor. Target verify is
 117:         # essentially a fixed sequence length prefill/extend with full cuda
 118:         # graph support. We can use it for target verify, or we can use it for
```

**读这段时抓住：**

- 自定义算法不是随便注册一个名字；它要暴露和 `SpeculativeAlgorithm` 相同的 `is_*` / `supports_*` 方法。
- `supports_overlap=False` 会影响 overlap schedule，可见 spec worker 和 scheduler 并不是松耦合插件。
- 如果插件算法要复用 EAGLE 分支，通常要提供自定义 `spec_class` 覆盖对应谓词。


### `create_worker`：算法名最后落到 worker 类

```python
# python/sglang/srt/speculative/spec_info.py:160-224
 160:         return self.is_eagle() or self.is_standalone()
 161: 
 162:     def handle_server_args(self, server_args: ServerArgs) -> None:
 163:         """Hook for per-algorithm server args mutation.
 164: 
 165:         In-place updated.
 166:         """
 167:         from sglang.srt.arg_groups.speculative_hook import (
 168:             _handle_dflash,
 169:             _handle_eagle_family,
 170:             _handle_frozen_kv_mtp,
 171:             _handle_ngram,
 172:         )
 173: 
 174:         if self.is_dflash():
 175:             _handle_dflash(server_args)
 176:         elif self.is_frozen_kv_mtp():
 177:             _handle_frozen_kv_mtp(server_args)
 178:         elif self.is_eagle() or self.is_standalone():
 179:             _handle_eagle_family(server_args)
 180:         elif self.is_ngram():
 181:             _handle_ngram(server_args)
 182: 
 183:     def get_num_tokens_per_bs_for_target_verify(
 184:         self, num_draft_tokens: int, is_draft_worker: bool
 185:     ) -> int:
 186:         # FIXME: Remove this after the forward mode refactor. Target verify is
 187:         # essentially a fixed sequence length prefill/extend with full cuda
 188:         # graph support. We can use it for target verify, or we can use it for
 189:         # other cases which is not target verify but fixed length prefill.
 190:         # Here, we expose this interface to allow the other use cases.
 191:         return num_draft_tokens
 192: 
 193:     def create_worker(
 194:         self, server_args: ServerArgs
 195:     ) -> Optional[Union[Type[BaseSpecWorker], Type[TpModelWorker], Type[NGRAMWorker]]]:
 196:         assert (
 197:             not self.is_none()
 198:         ), "Cannot create worker for NONE speculative algorithm."
 199: 
 200:         if self.is_dflash():
 201:             # V2 worker drives both overlap and non-overlap (scheduler runs it
 202:             # synchronously when overlap is disabled), same as EAGLE.
 203:             from sglang.srt.speculative.dflash_worker_v2 import DFlashWorkerV2
 204: 
 205:             return DFlashWorkerV2
 206: 
 207:         if self.is_frozen_kv_mtp():
 208:             # V2 worker drives both overlap and non-overlap (scheduler runs it
 209:             # synchronously when overlap is disabled), same as EAGLE.
 210:             from sglang.srt.speculative.frozen_kv_mtp_worker_v2 import (
 211:                 FrozenKVMTPWorkerV2,
 212:             )
 213: 
 214:             return FrozenKVMTPWorkerV2
 215: 
 216:         # EAGLE / EAGLE3 / STANDALONE / MULTI_LAYER always use the V2 worker,
 217:         # even with overlap disabled (scheduler drives it synchronously).
 218:         if self.is_eagle() and server_args.enable_multi_layer_eagle:
 219:             from sglang.srt.speculative.multi_layer_eagle_worker_v2 import (
 220:                 MultiLayerEagleWorkerV2,
 221:             )
 222: 
 223:             return MultiLayerEagleWorkerV2
 224: 
```

**读这段时抓住：**

- DFLASH、FROZEN_KV_MTP、EAGLE、STANDALONE、NGRAM 都在这里映射到不同 worker。
- EAGLE-family 默认使用 V2 worker，说明旧 spec worker 路径已经不是主线。
- 新增内建算法时，除了 server args hook，还要在这里定义 worker 映射。


## Worker 分层

`BaseSpecWorker` 管理 target worker 与 draft worker 的关系。EAGLE V2 worker 中可以看到：

- target forward 捕获 hidden states 或 logits。
- draft 根据 hidden/last token 生成候选。
- verify 让 target 检查候选。
- accept/reject kernel 计算接受长度和 bonus token。
- 更新 batch result，让 scheduler 像普通 decode 一样继续处理输出。


In [ ]:
for path in [
    "python/sglang/srt/speculative/base_spec_worker.py",
    "python/sglang/srt/speculative/eagle_worker_v2.py",
    "python/sglang/srt/speculative/ngram_worker.py",
    "python/sglang/srt/speculative/reject_sampling.py",
]:
    print("\n==", path)
    for lineno, kind, name in find_defs(path):
        if name in {"BaseSpecWorker", "EagleDraftWorkerBase", "EAGLEWorkerV2", "EagleDraftWorker", "NGRAMWorker", "draft", "verify", "draft_extend"}:
            print(f"{lineno:4d} {kind:16s} {name}")


### `BaseSpecWorker`：target worker 和 draft worker 的共同 contract

```python
# python/sglang/srt/speculative/base_spec_worker.py:274-324
 274: class BaseSpecWorker(ABC):
 275:     @property
 276:     @abstractmethod
 277:     def target_worker(self) -> TpModelWorker:
 278:         pass
 279: 
 280:     @property
 281:     @abstractmethod
 282:     def draft_worker(self) -> EagleDraftWorkerBase:
 283:         pass
 284: 
 285:     @property
 286:     def war_fastpath_runner(self):
 287:         # The runner that runs the step's LAST shared-buffer-reading phase --
 288:         # it owns the read-done event the scheduler's WAR barrier waits on.
 289:         # Default is the target runner; override if the last phase runs
 290:         # elsewhere (eagle's draft_extend runs on the draft runner).
 291:         return self.target_worker.model_runner
 292: 
 293:     @property
 294:     def spec_v2_attn_backends(self) -> tuple:
 295:         """Attn backends touched by spec_v2 forward; OR-ed by decide_needs_cpu_seq_lens.
 296:         Default returns target only; subclasses extend with draft backends."""
 297:         return (self.target_worker.model_runner.attn_backend,)
 298: 
 299:     @abstractmethod
 300:     def clear_cache_pool(self):
 301:         # TODO: move this abstract method to BaseTpWorker and call through self.model_runner
 302:         pass
 303: 
 304:     def alloc_memory_pool(self, **kwargs):
 305:         pass
 306: 
 307:     def init_attention_backends(self):
 308:         pass
 309: 
 310:     def init_cuda_graphs(self):
 311:         pass
 312: 
 313:     def on_verify_complete_cpu(
 314:         self, num_correct_drafts_per_req: list[int], batch_size: int = 0
 315:     ) -> None:
 316:         """Hook called after verify finishes and accept counts are on CPU.
 317: 
 318:         Default no-op. Adaptive-aware workers override this to feed the
 319:         controller without forcing a GPU→CPU sync in the worker hot path.
 320:         """
 321:         pass
 322: 
 323:     def activate_step_by_batch(self, batch_size: int) -> None:
 324:         """Activate the optimal adaptive step for the current batch size.
```

**读这段时抓住：**

- `target_worker` 和 `draft_worker` 是抽象属性；算法可以共享 target runner，也可以有独立 draft runner。
- `resolve_model_worker_batch` 默认按 target batch 解析，复杂算法会覆盖以适配 draft/verify 的 metadata。
- `finalize_after_verify` 是 accept counts 回到 CPU 后的 hook，用于更新 KV 或算法运行时状态。


### `EAGLEWorkerV2` 的 target/draft 关系

```python
# python/sglang/srt/speculative/eagle_worker_v2.py:937-1088
 937: class EAGLEWorkerV2(BaseSpecWorker):
 938:     def __init__(
 939:         self,
 940:         server_args: ServerArgs,
 941:         gpu_id: int,
 942:         tp_rank: int,
 943:         dp_rank: Optional[int],
 944:         moe_ep_rank: int,
 945:         attn_cp_rank: int,
 946:         moe_dp_rank: int,
 947:         nccl_port: int,
 948:         target_worker: TpModelWorker,
 949:     ):
 950:         # Parse arguments
 951:         self.server_args = server_args
 952:         self.topk = server_args.speculative_eagle_topk
 953:         self.speculative_num_steps = server_args.speculative_num_steps
 954:         self.speculative_num_draft_tokens = server_args.speculative_num_draft_tokens
 955:         self.tp_rank = tp_rank
 956:         self.gpu_id = gpu_id
 957:         self.device = server_args.device
 958:         self._target_worker = target_worker
 959:         self.page_size = server_args.page_size
      # 注：HiCache 所有 L1/L2/L3 KV 操作都按 page 对齐，page_size 是 load-back/prefetch 的基本粒度。
 960:         self.speculative_algorithm = SpeculativeAlgorithm.from_string(
      # 注：`self.speculative_algorithm` 接收 `SpeculativeAlgorithm.from_string` 的结果：把 CLI/参数字符串解析成内建或插件 speculative algorithm 对象。
 961:             server_args.speculative_algorithm
 962:         )
 963: 
 964:         # Override the context length of the draft model to be the same as the target model.
 965:         server_args.context_length = target_worker.model_runner.model_config.context_len
 966: 
 967:         self._draft_worker = EagleDraftWorker(
 968:             server_args,
 969:             gpu_id,
 970:             tp_rank,
 971:             dp_rank,
 972:             moe_ep_rank,
 973:             attn_cp_rank,
 974:             moe_dp_rank,
 975:             nccl_port,
 976:             target_worker,
 977:         )
 978: 
 979:         # Adaptive speculative
 980:         self.adaptive_controller: Optional[AdaptiveController] = None
 981:         if server_args.speculative_adaptive:
 982:             self.adaptive_controller = AdaptiveController(
 983:                 self,
 984:                 config_path=server_args.speculative_adaptive_config,
 985:             )
 986: 
 987:         # Some dummy tensors
 988:         self.num_new_pages_per_topk = torch.empty(
 989:             (), dtype=torch.int64, device=self.device
 990:         )
 991:         self.extend_lens = torch.empty((), dtype=torch.int64, device=self.device)
 992: 
 993:         self.plan_stream, self.plan_stream_ctx = _get_plan_stream(self.device)
 994: 
 995:     @property
 996:     def war_fastpath_runner(self):
 997:         # Per the base contract: the step's last shared-buffer-reading phase is
 998:         # draft_extend, which runs on the draft runner.
 999:         return self._draft_worker.draft_runner
1000: 
1001:     @property
1002:     def spec_v2_attn_backends(self) -> tuple:
1003:         # Every attn backend a spec_v2 forward touches; consumed by
1004:         # decide_needs_cpu_seq_lens to gate the seq_lens_cpu D2H.
1005:         return (
1006:             self._target_worker.model_runner.attn_backend,
1007:             self._draft_worker.draft_attn_backend,
1008:             self._draft_worker.draft_extend_attn_backend
1009:             or self._draft_worker.draft_runner.attn_backend,
1010:         )
1011: 
1012:     def alloc_memory_pool(
1013:         self,
1014:         memory_pool_config=None,
1015:         req_to_token_pool=None,
1016:         token_to_kv_pool_allocator=None,
1017:     ):
1018:         self._draft_worker.alloc_memory_pool(
1019:             memory_pool_config, req_to_token_pool, token_to_kv_pool_allocator
1020:         )
1021:         self.req_to_token_pool = req_to_token_pool
1022:         self.token_to_kv_pool_allocator = token_to_kv_pool_allocator
1023: 
1024:     def init_attention_backends(self):
1025:         self._draft_worker.init_attention_backends()
1026: 
1027:     def init_cuda_graphs(self):
1028:         self._draft_worker.init_cuda_graphs()
1029:         # Build adaptive runtime states after target and draft backends exist.
1030:         if self.adaptive_controller is not None:
1031:             with (
1032:                 self._draft_worker.draft_tp_context(
1033:                     self._draft_worker.draft_runner.tp_group
1034:                 ),
1035:                 speculative_moe_backend_context(),
1036:                 speculative_moe_a2a_backend_context(),
1037:             ):
1038:                 self.adaptive_controller.register(
1039:                     SpecRuntimeState(
1040:                         speculative_num_steps=self.speculative_num_steps,
1041:                         speculative_num_draft_tokens=self.speculative_num_draft_tokens,
1042:                         draft_attn_backend=self._draft_worker.draft_attn_backend,
1043:                         cuda_graph_runner=self._draft_worker.cuda_graph_runner,
1044:                         target_attn_backend=self._target_worker.model_runner.attn_backend,
1045:                         target_graph_runner=self._target_worker.model_runner.decode_cuda_graph_runner,
1046:                         draft_extend_attn_backend=self._draft_worker.draft_extend_attn_backend,
1047:                         cuda_graph_runner_for_draft_extend=self._draft_worker.cuda_graph_runner_for_draft_extend,
1048:                     )
1049:                 )
1050:                 self.adaptive_controller.init_states(
1051:                     cuda_graph_bs=(
1052:                         None
1053:                         if self.server_args.disable_cuda_graph
1054:                         else self.server_args.cuda_graph_bs_decode
1055:                     ),
1056:                 )
1057: 
1058:     @property
1059:     def target_worker(self):
1060:         return self._target_worker
1061: 
1062:     @property
1063:     def draft_worker(self):
1064:         return self._draft_worker
1065: 
1066:     def clear_cache_pool(self):
1067:         # allocator and kv cache pool are shared with target worker, which are cleared in scheduler
1068:         pass
1069: 
1070:     def forward_batch_generation(self, batch: ScheduleBatch, on_publish=None):
1071:         if batch.forward_mode.is_extend() or batch.is_extend_in_batch:
1072:             # Target prefill
1073:             target_capture_mode = (
1074:                 CaptureHiddenMode.NULL
1075:                 if self.speculative_algorithm.is_standalone()
1076:                 else CaptureHiddenMode.FULL
1077:             )
1078:             batch.capture_hidden_mode = target_capture_mode
1079:             batch_output = self.target_worker.forward_batch_generation(batch)
      # 注：`batch_output` 接收 `self.target_worker.forward_batch_generation` 的结果：用 target model 执行 verify 或普通生成 forward。
1080: 
1081:             # Spec_v2 convention: batch.seq_lens = length BEFORE this iter's tokens.
1082:             # Extend processed L prompt tokens; next verify iter expects same L.
1083:             batch_output.new_seq_lens = batch.seq_lens
1084:             # Publish before draft_extend so the fence is at target-end.
1085:             if on_publish is not None:
1086:                 on_publish(batch_output.new_seq_lens)
1087: 
1088:             # Draft prefill
```

**读这段时抓住：**

- `target_worker` 是主模型，`draft_worker` 包装 draft runner；两者共享或协调 KV allocator。
- prefill 时 target 先 forward，并以 hidden states/next token 驱动 draft extend。
- `capture_hidden_mode` 是 EAGLE 族的关键：draft 不是只看 token，还依赖 target hidden。


## Accept/reject 的正确性

Greedy verify 可以比较 draft token 与 target top-1；采样场景更复杂，需要 rejection sampling，确保输出分布仍等价于 target model。`reject_sampling.py` 里实现的是接受路径与最终 token 的概率修正。


In [ ]:
show_lines("python/sglang/srt/speculative/reject_sampling.py", 1, 45)
show_lines("python/sglang/srt/speculative/reject_sampling.py", 90, 125)


### `reject_sampling.py`：拒绝采样的核心分支

```python
# python/sglang/srt/speculative/reject_sampling.py:44-124
  44:     last_accepted_global_idx = root_global_idx
  45: 
  46:     num_accept = 0
  47: 
  48:     # Verification Loop
  49:     step = 1
  50:     continue_verifying = 1
  51: 
  52:     while (step < NUM_SLOTS) and (continue_verifying == 1):
  53:         draft_token = tl.load(cand_ptr_base + step * stride_cand_s)
  54: 
  55:         offset_prob = (
  56:             (pid * stride_tp_b)
  57:             + (cur_prob_row * stride_tp_s)
  58:             + (draft_token * stride_tp_v)
  59:         )
  60:         offset_draft = (
  61:             (pid * stride_dp_b)
  62:             + (cur_prob_row * stride_dp_s)
  63:             + (draft_token * stride_dp_v)
  64:         )
  65: 
  66:         p = tl.load(TargetProbs + offset_prob)
  67:         q = tl.load(DraftProbs + offset_draft)
  68: 
  69:         coin = tl.load(uni_ptr_base + (step - 1) * stride_uni_s)
  70: 
  71:         if coin * q < p:
  72:             num_accept += 1
  73:             cur_prob_row = step
  74:             tl.store(Predicts + last_accepted_global_idx, draft_token)
  75: 
  76:             curr_global_idx = tl.load(idx_ptr_base + step * stride_idx_s)
  77:             tl.store(
  78:                 AcceptIndex + pid * stride_idx_b + num_accept * stride_idx_s,
  79:                 curr_global_idx,
  80:             )
  81:             last_accepted_global_idx = curr_global_idx
  82: 
  83:             step += 1
  84:         else:
  85:             continue_verifying = 0
  86: 
  87:     tl.store(AcceptTokenNum + pid, num_accept)
  88: 
  89:     # Final Sampling
  90:     all_drafts_accepted = continue_verifying
  91:     coin_final = tl.load(UniformSamplesFinal + pid)
  92:     norm_sum = 0.0
  93: 
  94:     tp_base_ptr = TargetProbs + (pid * stride_tp_b) + (cur_prob_row * stride_tp_s)
  95:     # DraftProbs has only num_steps rows (TargetProbs has num_steps + 1). When
  96:     # all drafts are accepted cur_prob_row == num_steps is out of bounds for
  97:     # DraftProbs, but the all-accepted branch samples pure target p and never
  98:     # dereferences this pointer; on rejection cur_prob_row <= num_steps - 1.
  99:     dp_base_ptr_safe = DraftProbs + (pid * stride_dp_b) + (cur_prob_row * stride_dp_s)
 100: 
 101:     # Pass 1: Sum
 102:     for v_start in range(0, VOCAB_SIZE, BLOCK_V):
 103:         v_offsets = v_start + tl.arange(0, BLOCK_V)
 104:         mask = v_offsets < VOCAB_SIZE
 105: 
 106:         p_ptr = tp_base_ptr + v_offsets * stride_tp_v
 107:         p_val = tl.load(p_ptr, mask=mask, other=0.0)
 108: 
 109:         if all_drafts_accepted:
 110:             val = p_val
 111:         else:
 112:             q_ptr = dp_base_ptr_safe + v_offsets * stride_dp_v
 113:             q_val = tl.load(q_ptr, mask=mask, other=0.0)
 114:             diff = p_val - q_val
 115:             val = tl.where(diff > 0.0, diff, 0.0)
 116: 
 117:         norm_sum += tl.sum(val)
 118: 
 119:     # Pass 2: CDF. Degenerate residual (norm_sum == 0, i.e. p == q everywhere on
 120:     # rejection) leaves the cumsum at 0 <= target_u, so final_token falls back to
 121:     # VOCAB_SIZE - 1; acceptable since this case is numerically near-impossible.
 122:     target_u = coin_final * norm_sum
 123:     cum_sum = 0.0
 124:     final_token = VOCAB_SIZE - 1
```

**读这段时抓住：**

- kernel 逐步比较 draft token 在 target/draft 概率下是否可接受，接受则推进 `last_accepted_global_idx`。
- 一旦拒绝，最终 token 不再直接用 draft，而从修正后的 target-minus-draft 分布采样。
- 全部接受时还要采一个 bonus token，这就是 accept_lens 往往包含 bonus 的原因。


## KV cache 为什么是难点

Speculative decoding 不只是多生成几个 token。draft token 可能被拒绝，因此 KV 写入需要区分：

- draft KV 是否真实写入。
- target verify 使用哪段 cache loc。
- 接受后哪些 KV 要搬到 committed target KV。
- 拒绝后哪些临时 KV 要释放或覆盖。

这就是为什么 speculative 代码里会有 cache loc kernel、future map、verify buffer、draft/target attention backend 之间的同步。


In [ ]:
for path in [
    "python/sglang/srt/speculative/triton_ops/cache_locs.py",
    "python/sglang/srt/speculative/triton_ops/eagle.py",
    "python/sglang/srt/speculative/eagle_utils.py",
]:
    print("\n==", path)
    matches = [line for line in read_rel(path).splitlines() if "cache" in line.lower() or "accept" in line.lower()]
    for line in matches[:20]:
        print(line[:140])


## 小测试：算法名解析和插件保留名

这个测试只跑 Python enum/registry，不需要模型。


In [ ]:
from sglang.srt.speculative.spec_info import SpeculativeAlgorithm

for name in [None, "EAGLE", "eagle3", "standalone", "ngram", "none"]:
    algo = SpeculativeAlgorithm.from_string(name)
    print(name, "->", algo, "is_some=", algo.is_some(), "need_topk=", algo.need_topk() if hasattr(algo, "need_topk") else None)

try:
    SpeculativeAlgorithm.from_string("not-a-real-algo")
except ValueError as e:
    print("unknown name:", e)


## 贡献者注意点

- 新算法如果要进入主干，先明确它是否需要 draft KV、是否支持 overlap、是否需要 target hidden states。
- 所有 accept length 都会影响 scheduler 的输出 token 数、KV 提交数、metrics 和 finish 判断。
- 调试 spec decoding 时，先用 batch size 1、temperature 0、短 prompt 验证 token 序列，再放大到 overlap/batching。
